# Mock data challenge: measuring subhalo concentrations with flux ratios

Ten mock quadruply imaged quasars, all in a **fold** configuration: the source sits just inside a fold caustic, so two of the four images (a minimum and a saddle) sit close together on either side of the critical curve.

For a smooth lens, the two fold images have almost equal magnification. We measure this with the fold ratio

$$R_\mathrm{fold} = \frac{|\mu_1| - |\mu_2|}{|\mu_1| + |\mu_2|},$$

which goes to zero as the source approaches the fold. Dark matter subhalos near the images perturb the magnifications and scatter $R_\mathrm{fold}$ away from its smooth-model value. How much they scatter it depends on how many subhalos there are and how **concentrated** they are.

I made the "measured" $R_\mathrm{fold}$ values below with [pyHalo](https://github.com/dangilman/pyHalo) CDM subhalos, using a subhalo mass function normalization `SHMF_NORM = 0.2` and a secret value of the concentration normalization **`c0`**. Your job is to find `c0`.

## How to run this notebook in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amn3142/RESCEU_dm/blob/main/03_mock_data_challenge.ipynb)

1. Click the **Open in Colab** button above and sign in with your Google account if asked.
2. *(Optional)* Choose **File → Save a copy in Drive** if you want to keep your changes.
3. Run the cells **one at a time from the top** (Shift+Enter). When Colab says *"This notebook was not authored by Google"*, click **Run anyway**.
4. The first code cell installs `lenstronomy`, `pyHalo` and a few helpers, which takes about a minute.

Cells with a form on the right (sliders, boxes) let you change settings without editing code. Change a value, then re-run the cell.

**Troubleshooting**
- If Colab asks you to **Restart session**, click it and re-run the cells from the top. The install is skipped the second time.
- If you leave the notebook idle for a long time, Colab resets it. Re-run the cells from the top.

In [ ]:
# make sure lenstronomy and pyHalo are installed, otherwise install versions known to work together
try:
    import lenstronomy, pyHalo, multiprocess
except ImportError:
    %pip install lenstronomy==1.14.2 pyhalo==1.4.6 colossus mcfit multiprocess

import os
import time
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.LensModel.lens_model_extensions import LensModelExtensions

from pyHalo.PresetModels.cdm import CDM

In [ ]:
def plot_configuration(source_x=None, 
                       source_y=None,
                       x_img=None, 
                       y_img=None, 
                       mu=None,
                       crit_x=None, 
                       crit_y=None,
                       caus_x=None, 
                       caus_y=None,
                       axes=None, 
                       window=1.5, 
                       source_window=0.25, 
                       realization=None,
                       title=None):
    """Two-panel figure: source plane (left) and image plane (right)."""
    if axes is None:
        _, axes = plt.subplots(1, 2, figsize=(10.5, 5))
    ax_s, ax_i = axes

    # ---- source plane -------------------------------------------------------
    if caus_x is not None and caus_y is not None:
        for cx, cy in zip(caus_x, caus_y):
            ax_s.plot(cx, cy, color="crimson", lw=1.2)
    if source_x is not None and source_y is not None:
        ax_s.plot(source_x, source_y, marker="*", ms=16, color="k", ls="none")
    ax_s.set_xlim(-source_window, source_window)
    ax_s.set_ylim(-source_window, source_window)
    ax_s.set_aspect("equal")
    ax_s.set_xlabel(r"$\beta_x$ (arcsec)")
    ax_s.set_ylabel(r"$\beta_y$ (arcsec)")
    ax_s.set_title("source plane")

    # ---- image plane --------------------------------------------------------
    if crit_x is not None and crit_y is not None:
        for cx, cy in zip(crit_x, crit_y):
            ax_i.plot(cx, cy, color="crimson", lw=1.2)
    if x_img is not None and y_img is not None:
        x_img, y_img = np.atleast_1d(x_img), np.atleast_1d(y_img)
        size = 60.0 if mu is None else 30 + 12 * np.abs(np.asarray(mu))
        ax_i.scatter(x_img, y_img, s=size, c="g", zorder=5, alpha=0.25)
        for k, (xi, yi) in enumerate(zip(x_img, y_img)):
            label = "ABCDEFGH"[k] if k < 8 else str(k)
            if mu is not None:
                label += f"  $\\mu$={np.asarray(mu)[k]:.1f}"
            ax_i.annotate(label, (xi, yi), textcoords="offset points",
                          xytext=(8, 6), fontsize=9)
    ax_i.plot(0, 0, marker="+", ms=10, color="0.4")
    ax_i.set_xlim(-window, window)
    ax_i.set_ylim(-window, window)
    ax_i.set_aspect("equal")
    ax_i.set_xlabel(r"$\theta_x$ (arcsec)")
    ax_i.set_ylabel(r"$\theta_y$ (arcsec)")
    ax_i.set_title("image plane")

    if realization is not None:
        for halo in realization.halos:
            s = 40 * (halo.bound_mass / 10 ** 8) ** 0.5
            ax_i.scatter(halo.x, halo.y, color='k', alpha=0.5, s=s)
    
    if title:
        axes[0].figure.suptitle(title, y=1.02)
    return axes

def critical_curves_and_caustics(kwargs_lens, lens_model,
                                 compute_window=3.0, grid_scale=0.008):
    """Return (crit_x_list, crit_y_list, caustic_x_list, caustic_y_list)."""
    ext = LensModelExtensions(lens_model)
    return ext.critical_curve_caustics(kwargs_lens, 
                                       compute_window=compute_window,
                                       grid_scale=grid_scale)

    
def statistic_from_magnifications(mu, image_config, index_list):

    if image_config == 'FOLD':
        r_fold = (mu[index_list[0]] - mu[index_list[1]]) / (mu[index_list[0]] + mu[index_list[1]])
        return r_fold
    else:
        r_cusp = (mu[index_list[0]] - mu[index_list[1]] + mu[index_list[2]]) / (mu[index_list[0]] + mu[index_list[1]] + mu[index_list[2]])
        return r_cusp

## Mock lens macromodels

In [ ]:
kwargs_lens_macro_1 = [{'theta_E':  0.9, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.2, # ellipticity in cartesian coordinates
                      'e2': 0.1, # ellipticity in cartesian coords
                      'gamma': 2.1 # logarithmic power law slope
                     }]

kwargs_lens_macro_2 = [{'theta_E':  1.1, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': -0.05, # ellipticity in cartesian coordinates
                      'e2': -0.1, # ellipticity in cartesian coords
                      'gamma': 1.9 # logarithmic power law slope
                     }]
kwargs_lens_macro_3 = [{'theta_E':  1.2, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': -0.05, # ellipticity in cartesian coordinates
                      'e2': 0.14, # ellipticity in cartesian coords
                      'gamma': 2.1 # logarithmic power law slope
                     }]
kwargs_lens_macro_4 = [{'theta_E':  0.8, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.3, # ellipticity in cartesian coordinates
                      'e2': 0.1, # ellipticity in cartesian coords
                      'gamma': 2.05 # logarithmic power law slope
                     }]
kwargs_lens_macro_5 = [{'theta_E':  1.1, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.05, # ellipticity in cartesian coordinates
                      'e2': 0.1, # ellipticity in cartesian coords
                      'gamma': 2.15 # logarithmic power law slope
                     }]
kwargs_lens_macro_6 = [{'theta_E':  1.0, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.05, # ellipticity in cartesian coordinates
                      'e2': 0.1, # ellipticity in cartesian coords
                      'gamma': 2.05 # logarithmic power law slope
                     }]
kwargs_lens_macro_7 = [{'theta_E':  1.2, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.05, # ellipticity in cartesian coordinates
                      'e2': -0.15, # ellipticity in cartesian coords
                      'gamma': 2.2 # logarithmic power law slope
                     }]
kwargs_lens_macro_8 = [{'theta_E':  0.9, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.0, # ellipticity in cartesian coordinates
                      'e2': -0.1, # ellipticity in cartesian coords
                      'gamma': 2.1 # logarithmic power law slope
                     }]
kwargs_lens_macro_9 = [{'theta_E':  1.1, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.2, # ellipticity in cartesian coordinates
                      'e2': -0.1, # ellipticity in cartesian coords
                      'gamma': 2.1 # logarithmic power law slope
                     }]
kwargs_lens_macro_10 = [{'theta_E':  1.05, # Einstein radius
                      'center_x': 0.0, # mass centroid
                      'center_y': 0.0, # mass centroid
                      'e1': 0.1, # ellipticity in cartesian coordinates
                      'e2': 0.15, # ellipticity in cartesian coords
                      'gamma': 2.08 # logarithmic power law slope
                     }]

In [ ]:
# source positions (arcsec). Placed relative to each lens's astroid caustic:
# folds lie at phi +/- 45 deg from the ellipticity position angle
# phi = 0.5*arctan2(e2, e1), at a radius of about 0.6 * theta_E * |e|.
source_x_1,  source_y_1  =  0.10, -0.055   # FOLD
index_list_1 = [1,2]
source_x_2,  source_y_2  = -0.04,  -0.065   # FOLD
index_list_2 = [1,2]
source_x_3,  source_y_3  =  0.015, -0.10   # FOLD
index_list_3 = [1,2]
source_x_4,  source_y_4  =  0.06,   0.13    # FOLD
index_list_4 = [1,2]
source_x_5,  source_y_5  = -0.06,  0.016   # FOLD
index_list_5 = [1,2]
source_x_6,  source_y_6  =  0.010,  0.062   # FOLD
index_list_6 = [1,2]
source_x_7,  source_y_7  = -0.10, -0.015   # FOLD
index_list_7 = [1,2]
source_x_8,  source_y_8  = -0.051,  0.000   # FOLD
index_list_8 = [1,2]
source_x_9,  source_y_9  = -0.12, -0.07   # FOLD
index_list_9 = [1,2]
source_x_10, source_y_10 = -0.085,  0.069   # FOLD
index_list_10 = [1,2]

lens_model_list_macro = ['EPL']

mocks = []
for i in range(1, 11):
    mocks.append({'index': i,
                  'kwargs_lens_macro': eval(f'kwargs_lens_macro_{i}'),
                  'source_x': eval(f'source_x_{i}'),
                  'source_y': eval(f'source_y_{i}'),
                  'index_list': eval(f'index_list_{i}'),
                  'image_config': 'FOLD'})

## Lensing with subhalos

`make_realization` draws one random population of CDM subhalos with pyHalo, and `solve_lens` finds the images and their magnifications for one mock lens, with or without subhalos.

To keep things fast, we only render subhalos more massive than $10^{8.5}\,M_\odot$ (at infall) inside a cone 3 arcsec across, centred on the lens. Flux ratios are dominated by the more massive subhalos close to the images, so these cuts barely change the result.

In [ ]:
def solve_lens(m, realization=None):
    # image positions and magnifications for one mock lens, with or without a subhalo realization
    lens_model_list, kwargs_lens = list(lens_model_list_macro), list(m['kwargs_lens_macro'])
    if realization is not None:
        lens_model_list_halos, _, kwargs_halos, _ = realization.lensing_quantities()
        lens_model_list += lens_model_list_halos
        kwargs_lens += kwargs_halos
    lens_model = LensModel(lens_model_list)
    ximg, yimg = LensEquationSolver(lens_model).image_position_from_source(
        m['source_x'], m['source_y'], kwargs_lens, search_window=3.0)
    mu = lens_model.magnification(ximg, yimg, kwargs_lens)
    return ximg, yimg, mu, lens_model, kwargs_lens


def make_realization(c0, SHMF_NORM=0.2, seed=0):
    # one random population of CDM subhalos; the seed fixes their masses and positions
    np.random.seed(seed)
    z_lens, z_source = 0.5, 2.0
    kwargs_concentration_model_subhalos = {'c0': c0,
                                           'beta': 0.8,
                                           'zeta': 0.42}
    cdm_realization = CDM(z_lens,
                          z_source,
                          sigma_sub=SHMF_NORM,
                          LOS_normalization=0.0,
                          log_mlow=8.5,
                          log_mhigh=10.0,
                          cone_opening_angle_arcsec=3.0,
                          concentration_model_subhalos='PEAK_HEIGHT_POWERLAW',
                          kwargs_concentration_model_subhalos=kwargs_concentration_model_subhalos)
    return cdm_realization.filter_bound_mass(10 ** 6)

## The data

These are the ten measured fold ratios. The smooth macromodel alone doesn't give $R_\mathrm{fold} = 0$: each source sits a finite distance from its fold, so each lens has its own smooth-model value $R_\mathrm{macro}$. The **subhalo signal** is the difference $\Delta R = R_\mathrm{measured} - R_\mathrm{macro}$.

In [ ]:
R_FOLD_MEASURED = np.array([-0.2131, 0.0277, 0.0193, 0.0991, 0.1796, 0.0399, -0.0065, 0.1951, 0.0999, 0.1213])

R_MACRO = []
for m in mocks:
    ximg, yimg, mu, _, _ = solve_lens(m)
    R_MACRO.append(statistic_from_magnifications(np.abs(mu), m['image_config'], m['index_list']))
R_MACRO = np.array(R_MACRO)
DELTA_R_MEASURED = R_FOLD_MEASURED - R_MACRO

print('lens   R_measured   R_macro   Delta R')
for j in range(len(mocks)):
    print(f'{j + 1:3d}     {R_FOLD_MEASURED[j]:+.3f}     {R_MACRO[j]:+.3f}    {DELTA_R_MEASURED[j]:+.3f}')

xs = np.arange(1, len(mocks) + 1)
f, ax = plt.subplots(figsize=(9, 4))
ax.bar(xs - 0.2, R_FOLD_MEASURED, 0.4, label='measured')
ax.bar(xs + 0.2, R_MACRO, 0.4, label='smooth macromodel only')
ax.axhline(0, color='k', lw=0.8)
ax.set_xticks(xs)
ax.set_xlabel('mock lens')
ax.set_ylabel(r'$R_\mathrm{fold}$')
ax.legend()
plt.show()

Here are the ten lenses without any subhalos. Images B and C are the fold pair.

In [ ]:
for j, m in enumerate(mocks):
    ximg, yimg, mu, lens_model, kwargs_lens = solve_lens(m)
    crit_x, crit_y, caus_x, caus_y = critical_curves_and_caustics(kwargs_lens, lens_model)
    plot_configuration(source_x=m['source_x'], source_y=m['source_y'], x_img=ximg, y_img=yimg, mu=mu,
                       crit_x=crit_x, crit_y=crit_y, caus_x=caus_x, caus_y=caus_y,
                       title=f'Mock lens #{j + 1}    $R_\\mathrm{{fold}}$ measured: {R_FOLD_MEASURED[j]:.2f}')
    plt.show()

## Step 1: explore one subhalo realization

This cell draws one random population of CDM subhalos (black dots, sized by bound mass) and puts it in one of the mock lenses. The critical curves now wiggle around the subhalos.

Try:
- the same `seed` with different `c0`: the subhalos stay in the same places, but their concentrations change. How does $R_\mathrm{fold}$ respond?
- different `seed` values at fixed `c0`: how much does $R_\mathrm{fold}$ scatter from one realization to the next?
- a subhalo that lands right on top of an image. Sometimes it creates extra images or destroys one.

In [ ]:
lens_number = 1  #@param {type:"slider", min:1, max:10, step:1}
c0 = 30  #@param {type:"slider", min:2, max:100, step:1}
seed = 1  #@param {type:"integer"}
SHMF_NORM = 0.2

m = mocks[lens_number - 1]
realization = make_realization(c0, SHMF_NORM, seed)
ximg, yimg, mu, lens_model, kwargs_lens = solve_lens(m, realization)
crit_x, crit_y, caus_x, caus_y = critical_curves_and_caustics(kwargs_lens, lens_model, grid_scale=0.02)
plot_configuration(source_x=m['source_x'], source_y=m['source_y'], x_img=ximg, y_img=yimg, mu=mu,
                   crit_x=crit_x, crit_y=crit_y, caus_x=caus_x, caus_y=caus_y, realization=realization,
                   title=f'Mock lens #{lens_number}, c0 = {c0}, seed = {seed}')
plt.show()

print(f'{len(realization.halos)} subhalos above 10^6 M_sun bound mass, {len(ximg)} images')
if len(ximg) == 4:
    R = statistic_from_magnifications(np.abs(mu), m['image_config'], m['index_list'])
    print(f'R_fold = {R:+.3f}    (smooth model: {R_MACRO[lens_number - 1]:+.3f}, measured: {R_FOLD_MEASURED[lens_number - 1]:+.3f})')
else:
    print('Not a quad any more, so R_fold is undefined for this realization.')

## The forward model

`forward_model_population(c0, seed)` puts one subhalo realization in each of the ten lenses and returns the ten simulated $\Delta R$ values. It runs the lenses in parallel.

- Each call with the **same `seed`** gives the same subhalo masses and positions. Use different seeds to get new realizations.
- A realization that doesn't give exactly four images returns `nan`. Skip those with `np.isfinite` or `np.nanmean`, for example.

In [ ]:
from multiprocess import Pool

N_THREADS = os.cpu_count()


def _forward_model_single(args):
    # one mock lens, one halo realization
    m, R_macro, c0, SHMF_NORM, seed = args
    realization = make_realization(c0, SHMF_NORM, seed)
    ximg, yimg, mu, _, _ = solve_lens(m, realization)
    if len(ximg) != 4:
        return np.nan  # a subhalo created or destroyed an image
    return statistic_from_magnifications(np.abs(mu), m['image_config'], m['index_list']) - R_macro


def forward_model_population(c0, SHMF_NORM=0.2, seed=0):
    # returns the simulated Delta R = R_fold - R_macro for all ten mock lenses
    jobs = [(m, R_MACRO[j], c0, SHMF_NORM, 1000 * seed + j) for j, m in enumerate(mocks)]
    with Pool(N_THREADS) as pool:
        return np.array(pool.map(_forward_model_single, jobs))


t = time.time()
print(np.round(forward_model_population(c0=10, seed=0), 3))
print(f'took {time.time() - t:.0f} s on {N_THREADS} cores')

## Step 2: simulate populations for a grid of `c0` values

For each `c0` in `c0_grid`, run the forward model `n_seeds` times. This gives `10 * n_seeds` simulated $\Delta R$ values per `c0`.

On Colab (2 cores), the default settings take about 2 minutes; higher `c0` is slower because more subhalos survive tidal stripping. More seeds give smoother histograms but take proportionally longer. If you work in a group, split the work between you and share the results.

The same seeds are used for every `c0`, so the differences between `c0` values come from the concentrations and not from random luck.

In [ ]:
c0_grid = [5, 10, 20, 40, 80]
n_seeds = 3  #@param {type:"slider", min:1, max:20, step:1}

simulations = {}  # c0 -> array of simulated Delta R values
t = time.time()
for c0 in c0_grid:
    simulations[c0] = np.concatenate([forward_model_population(c0, seed=s) for s in range(n_seeds)])
    print(f'c0 = {c0:3d}: {np.isfinite(simulations[c0]).sum()} usable realizations '
          f'({time.time() - t:.0f} s so far)')

## Step 3: compare simulations and data

The histograms show the distribution of simulated $\Delta R$ for each `c0`. The black ticks are the ten measured values. Which `c0` looks most like the data?

In [ ]:
bins = np.linspace(-0.6, 0.6, 31)
f, ax = plt.subplots(figsize=(9, 5))
for c0 in c0_grid:
    ax.hist(simulations[c0][np.isfinite(simulations[c0])], bins=bins, density=True, histtype='step', lw=2,
            label=f'c0 = {c0}')
for d in DELTA_R_MEASURED:
    ax.axvline(d, ymax=0.08, color='k', lw=2)
ax.set_xlabel(r'$\Delta R = R_\mathrm{fold} - R_\mathrm{macro}$')
ax.set_ylabel('probability density')
ax.legend()
plt.show()

print('width (std) of Delta R:')
print(f'  measured:  {np.std(DELTA_R_MEASURED):.3f}')
for c0 in c0_grid:
    print(f'  c0 = {c0:3d}:  {np.nanstd(simulations[c0]):.3f}')

# YOUR MISSION, SHOULD YOU CHOOSE TO ACCEPT IT:

## What value of `c0` did I use to create the mock data? Hint: I used `SHMF_NORM = 0.2`.

Turn the comparison above into a number, with an uncertainty. Some ideas:
- **Likelihood:** for each `c0`, estimate the probability density of $\Delta R$ from the simulations (for example with `scipy.stats.gaussian_kde`). Evaluate it at the ten measured values and multiply. Plot the log-likelihood against `c0`.
- **Approximate Bayesian computation (ABC):** pick a summary statistic (for example the standard deviation of the ten $\Delta R$ values). For many simulated populations, keep the `c0` values whose summary statistic is close to the measured one.
- Use a finer `c0_grid` near the best value, or more `n_seeds`, to check that your answer is stable.

good luck.

In [ ]:
# YOUR CODE HERE

In [ ]:
#@title Reference solution (double-click to show the code)
from scipy.stats import gaussian_kde

log_like = []
for c0 in c0_grid:
    sims = simulations[c0][np.isfinite(simulations[c0])]
    log_like.append(np.sum(np.log(gaussian_kde(sims)(DELTA_R_MEASURED) + 1e-10)))
log_like = np.array(log_like)

f, ax = plt.subplots(figsize=(7, 4))
ax.plot(c0_grid, log_like - log_like.max(), 'o-')
ax.set_xscale('log')
ax.set_xlabel('c0')
ax.set_ylabel(r'$\ln L - \ln L_\mathrm{max}$')
ax.set_title('likelihood of the measured fold ratios')
plt.show()
print('best c0 on the grid:', c0_grid[np.argmax(log_like)])

## Things to think about

- Why do more concentrated subhalos cause larger flux-ratio anomalies? *(Hint: what sets the magnification perturbation, the subhalo's total mass or the mass in its center?)*
- In Step 1, the number of subhalos changes with `c0` even at a fixed seed. Why? *(Hint: `filter_bound_mass` keeps subhalos whose bound mass after tidal stripping is above $10^6\,M_\odot$.)*
- I told you `SHMF_NORM = 0.2`. If you didn't know it, could you still measure `c0`? Try a few `SHMF_NORM` values in the forward model and see whether they are degenerate with `c0`.
- How would measurement errors on the flux ratios (a few percent in real data) change your answer?
- Which lenses carry the most information, and why?